In [1]:
import os
import sys
dir_path = "/qumulo/shared_data/aofei_summer/RegTok/RegLLM"
sys.path.insert(0, dir_path)
os.environ['CUDA_VISIBLE_DEVICES'] = "6"
os.environ["HF_HUB_CACHE"]="/qumulo/shared_data/aofei_summer/LLMs"
from llava.eval.cli_v1 import RegLLMChatbot

/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model_dir = "/qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/i2t_instruct_region_final"
# model_dir = "/qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/i2t_instruct_region03"
model_args = {
        "model_name_or_path": "Qwen/Qwen3-8B",
        "pretrained_llm_path": model_dir,
        "peft_path": None,
        "regtok_config_path": "/qumulo/shared_data/aofei_summer/RegTok/source/tokenizer/regtok_config.yaml",
        "regtok_weight_path": "/qumulo/shared_data/aofei_summer/intern_records/RegTok/checkpoints/RegTok_pipeline_full_wo_quant/002-RegTok/checkpoints/0079280.pt",
        "use_regtok": True,
        "mm_vision_vq_type": "RegTok",
        "use_region_tokens": False,
        "vision_tower": "/qumulo/shared_data/aofei_summer/CLIPs/unimed_clip_vit_l14.pt",
        "mm_use_im_start_end": False,
        "mm_use_im_patch_token": True,
        "mm_vision_select_feature": "patch",
        "mm_patch_merge_type": "flat",
        "mm_projector_type": "mlp2x_gelu",
        "pretrain_mm_mlp_adapter": None,
        "mm_vision_select_layer": -1,
        "use_region_tokens": True,
        "use_sep_proj": False,
        "output_segmentation": False,
        "use_moe": False,
        "use_seg_loss": False
        
    }

In [3]:
bot = RegLLMChatbot(model_dir, model_args=model_args, device="cuda")

loading model from /qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/i2t_instruct_region_final


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 60.81it/s]

load vision tower!


Number of stacks: 1
Upsample mode: conv
tokenflow load from: /qumulo/shared_data/aofei_summer/intern_records/RegTok/checkpoints/RegTok_pipeline_full_wo_quant/002-RegTok/checkpoints/0079280.pt
tokenflow model load success!!
mm_projector parameters unfrozen.
pre loading complete!
loading from /qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/i2t_instruct_region_final
[] missing: []


In [4]:
# bot = RegLLMChatbot(model_dir, model_args=model_args, device="cuda")

In [5]:
bot.inference("What modality is used to take this image?", images="/qumulo/shared_data/aofei_summer/data/evaluation/imgs/xmlab102/source.jpg")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


(["assistant\nThe image is a computed tomography (CT) scan of the chest. CT imaging uses X-rays to create detailed, cross-sectional images of the body's internal structures, allowing for comprehensive evaluation of organs and tissues within the thoracic cavity."],
 tensor([[151644,  77091,    198,    785,   2168,    374,    264,  24182,  10180,
            5696,    320,   1162,      8,   8569,    315,    279,  15138,     13,
           18572,  31658,   5711,   1599,  81717,    311,   1855,  11682,     11,
            5312,  96219,   5335,    315,    279,   2487,    594,   5306,  14389,
              11,  10693,    369,  15817,  16460,    315,  35753,    323,  38781,
            2878,    279,  72733,  93157,  55329,     13, 151645]],
        device='cuda:0'))

In [14]:
bot.inference("What modality is used to take this image?", images="/qumulo/shared_data/aofei_summer/data/evaluation/imgs/xmlab102/source.jpg")

["assistant\nThe image is a computed tomography (CT) scan of the chest. CT imaging uses X-rays to create detailed, cross-sectional images of the body's internal structures, allowing for comprehensive evaluation of various medical conditions affecting the lungs and other thoracic organs."]

In [7]:
import json
from tqdm import tqdm
model_name = "instruct_region_1105_sampling"
dataset_name = "VQA-RAD"
# question_file = "/qumulo/shared_data/aofei_summer/data/evaluation/test_processed.json"
# answers_file = f"/qumulo/shared_data/aofei_summer/data/evaluation/inference/answers_{model_name}.jsonl"
# image_folder = "/qumulo/shared_data/aofei_summer/data/evaluation/imgs"

question_file = f"/qumulo/shared_data/aofei_summer/data/evaluation_data/{dataset_name}/test.json"
answers_file = f"/qumulo/shared_data/aofei_summer/data/evaluation_data/{dataset_name}/inference/answers_{model_name}.jsonl"
image_folder = f"/qumulo/shared_data/aofei_summer/data/evaluation_data/{dataset_name}/image"

In [8]:
questions = json.load(open(os.path.expanduser(question_file), "r"))
answers_file = os.path.expanduser(answers_file)
os.makedirs(os.path.dirname(answers_file), exist_ok=True)
ans_file = open(answers_file, "w")
for line in tqdm(questions):

    # idx = line["qid"]
    # question = line["question"] # ['value'].split('\n')[0]
    # gt_ans = line["answer"] # ['value']      
    # image_file = line["img_name"]

    idx = line["id"]
    question = line["conversations"][0]["value"] # ['value'].split('\n')[0]
    gt_ans = line['conversations'][1]['value'] # ['value']
    image_file = line["image"]

    qs = question
    
    image_file = os.path.join(image_folder, image_file)
    ans = bot.inference(qs, image_file)[0][0]
    ans = ans.replace("assistant\n", "").strip()

    ans_file.write(json.dumps({"question_id": idx,
                                   "prompt": qs,
                                   "text": ans,
                                   "gt_ans": gt_ans,
                                   "metadata": {}}) + "\n")
    ans_file.flush()
ans_file.close()

100%|██████████| 451/451 [13:29<00:00,  1.80s/it]


In [9]:
(79.92 + 47.16) / 2

63.54